# Exercise 4: five years of the eclipsing binary ★★★★☆

*Covers notebook 3 and the ArgusSim survey tutorial.*

Eleven nights of prototype data found the binary's period only with Gaia's
help, however the one-cycle-per-day aliases were nearly as strong as the true peak.
What would the Array's survey do?

**Goal:** turn the prototype's folded light curve into a model, inject it
into the simulated survey at the binary's position, and search the
simulated light curve for the period.

This needs `argus_sim` (`pip install ./argus/argus_survey_model`).

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import AsinhStretch, ImageNormalize, ZScaleInterval

import argex
from argex import sky, times

warnings.filterwarnings("ignore", category=RuntimeWarning)  # all-NaN slices
GREEK = {"beta": "β", "rho": "ρ"}  # band names in plot labels
plt.rcParams.update({"figure.dpi": 110, "image.origin": "lower"})

# The staged sample. Set ARGUS_SAMPLE to read another copy, such as a download.
SAMPLE = Path(os.environ.get("ARGUS_SAMPLE", "~/s3/schmidt-observatory-system/argus/prototype_v1")).expanduser()
assert (SAMPLE / "manifest.json").is_file(), f"no sample at {SAMPLE}: set ARGUS_SAMPLE"

from astropy.timeseries import LombScargle

TARGET = dict(name="Gaia DR3 240827852899491328", ra=47.786654, dec=43.945067,
              bp_rp=0.919, gaia_frequency=1.2673242)
ra, dec = TARGET["ra"], TARGET["dec"]
P = 1 / TARGET["gaia_frequency"]   # d

In [ ]:
# The simulated 5-year survey (argus_survey_model/). Set ARGUS_SIM_STORE to read another copy.
import logging

from argus_sim.lightcurve import SurveyRun, bin_lightcurve

logging.getLogger("argus_sim").setLevel(logging.WARNING)
SIM_STORE = Path(os.environ.get(
    "ARGUS_SIM_STORE", "~/s3/schmidt-observatory-system/argus/sim/argsim_5yr_beta_rho_uniform_v5.4/store")).expanduser()
run = SurveyRun(str(SIM_STORE))

In [ ]:
from astropy.stats import SigmaClip
from astropy.table import Table
from photutils.aperture import ApertureStats, CircularAnnulus, CircularAperture, aperture_photometry

EXPTIME = 60.0                                # s, every frame
SEGMENT_PIXEL, DETECTOR_PIXEL = 0.879, 1.012  # arcsec
R = 4 * DETECTOR_PIXEL / SEGMENT_PIXEL        # the zeropoints' 4.05" aperture, in segment pixels
ANNULUS = (R + 3, 15.5)                       # what fits in a 32-pixel cutout


def aperture_flux(cube, x, y, radius=R, annulus=ANNULUS):
    # Background-subtracted flux (e-/s) and its error in each plane of a cube,
    # with the background the sigma-clipped median of the annulus.
    aperture = CircularAperture((x, y), r=radius)
    ring = CircularAnnulus((x, y), r_in=annulus[0], r_out=annulus[1])
    clip = SigmaClip(sigma=3.0)
    flux, err = np.full(len(cube), np.nan), np.full(len(cube), np.nan)
    for k, plane in enumerate(np.asarray(cube, np.float64)):
        sky_ = ApertureStats(plane, ring, sigma_clip=clip)
        phot = aperture_photometry(plane - sky_.median, aperture,
                                   error=np.full(plane.shape, sky_.std), method="exact")
        flux[k] = phot["aperture_sum"][0]
        # the sky noise from the annulus, plus the source's photon noise
        err[k] = np.hypot(phot["aperture_sum_err"][0], np.sqrt(max(flux[k], 0) / EXPTIME))
    return flux, err


def light_curve(frames, epochs, flux, err, zp, flags=None):
    # A table of calibrated magnitudes; a frame is good when it is calibrated,
    # has a positive flux and (with DQ flags) none of argex.dq.BAD under the aperture.
    mag = zp - 2.5 * np.log10(flux)
    good = np.isfinite(mag)
    if flags is not None:
        good &= (flags & int(argex.dq.BAD)) == 0
    return Table({
        "epoch": epochs, "mjd": times.to_mjd(epochs),
        "frame_key": [f.frame_key for f in frames],
        "flux": flux, "flux_err": err, "zp": zp,
        "mag": mag, "mag_err": 2.5 / np.log(10) * err / flux,
        "dq": np.zeros(len(flux), np.uint32) if flags is None else flags, "good": good,
    })

## 1. A model from the prototype

Measure the target in each band as in notebook 3 (the calibrated
magnitudes are enough), fold them at the Gaia period, and take the median
in 40 phase bins. Some bins have no data; interpolate across them.

ArgusSim's band names are `beta` and `rho`. Write
`model(mjd, band)` returning the AB magnitude for arrays of MJD and band
names, as ArgusSim's `inject()` expects.

In [ ]:
s = argex.read(SAMPLE / "seg_391749" / "minipix" / "minipix_0595.fits")
x, y = s.radec_to_pixel(ra, dec)
gr = (TARGET["bp_rp"] - 0.355) / 1.198
curves = {}   # ArgusSim band name -> (phase bin centers, median mag)
...

def model(mjd, band):
    ...

## 2. Inject and bin

Get the simulated visits at the target over the whole survey, inject
the model (`run.inject(vis, model, rng=...)`), and bin it per 16-minute
pointing (`bin_lightcurve(lc, by="ratchet")`).

In [ ]:
vis = ...
lc = ...
binned = ...

## 3. The period

Notebook 3's single-sine periodogram peaked at twice the orbital frequency:
two eclipses per orbit look most like a sine with two cycles per orbit. Give
the fit a second harmonic, `LombScargle(..., nterms=2)`, so that the orbital
frequency can win wherever the two eclipses differ.

Run it on the β pointing bins over 0.1–6 cycles per day. How close is the
peak period to Gaia's, and how tall are the one-cycle-per-day aliases next to
the highest peak? Then compare the power at the orbital frequency and at twice
it, in β and in ρ.

In [ ]:
def periodogram(band):
    ...

freq, power = periodogram("beta")
...

## 4. Questions

* Compare with notebook 3's periodogram from eleven prototype nights. What
  removed the ambiguity: the number of nights, the length of the baseline,
  or the coverage through each night?
* In ρ the orbital frequency and twice it nearly tie. Why? Look at the two
  eclipses of the model in each band, and at which phases of it the
  prototype never covered (section 1).
* Repeat with one month, then one season, of simulated data (cut `vis` on
  `mjd`). When does the true period win, and how does the period's precision
  grow with the baseline?
* The model is the prototype's own light curve, gaps interpolated. How would
  a gap at an eclipse change what you find?